# SADAM: run, test and reproduce

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alilouati/sadam-gaussian/blob/main/SADAM_demo.ipynb)

Companion notebook for *"Gaussian-Based Moment-State Initialization for Adaptive Optimization in Machine Learning"*
(A. Louati, H. Louati, E. Kariri; IEEE Access, under review). Code: <https://github.com/alilouati/sadam-gaussian>

SADAM is Adam with random initial moment states, $m_0 \sim \mathcal{N}(0,\sigma_m^2)$ and $v_0 = z^2,\ z \sim \mathcal{N}(0,\sigma_v^2)$;
all later updates are Adam's. This notebook

1. installs the code and runs the test suite,
2. shows SADAM in a few lines and its implicit warm-up effect,
3. regenerates the paper's tables and figures from the raw logs of all 287 runs,
4. lets you train Adam and SADAM yourself and change $\sigma_m$, $\sigma_v$,
5. (optional) re-runs real MNIST training and compares it with the logged result.

A CPU runtime is enough. Use **Runtime → Run all**; the whole notebook takes about 3–5 minutes.

## 0. Setup

In [ ]:
import os, subprocess, sys
REPO = "https://github.com/alilouati/sadam-gaussian.git"
if not os.path.exists("code/optim.py"):                     # not already inside the repository
    if not os.path.exists("sadam-gaussian"):
        subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
    os.chdir("sadam-gaussian")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "pytest"], check=True)
sys.path.insert(0, os.path.abspath("code"))
print("Ready. Working directory:", os.getcwd())

## 1. Run the test suite
16 tests: Gaussian initialization, the closed form of Lemma 1 at $t=1$ (factors 9 and 999), $\sigma_m=\sigma_v=0$ recovers Adam exactly,
finite-difference check of every layer, all ten optimizers train a small network, and the paper's tables are regenerated exactly from the logs.

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-v", "-p", "no:cacheprovider"], capture_output=True, text=True)
print(r.stdout[-5000:]); print(r.stderr[-2000:])
assert r.returncode == 0, "some tests failed"

## 2. SADAM in a few lines
A self-contained NumPy version (the experiments use `code/optim.py`, which implements SADAM and all baselines).

In [ ]:
import numpy as np

class SADAM:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, sigma_m=0.01, sigma_v=0.01, seed=0):
        rng = np.random.default_rng(seed)
        self.params, self.lr, (self.b1, self.b2), self.eps, self.t = params, lr, betas, eps, 0
        self.m = [rng.normal(0, sigma_m, p.shape) for p in params]          # m0 ~ N(0, sigma_m^2)
        self.v = [rng.normal(0, sigma_v, p.shape) ** 2 for p in params]     # v0 = z^2,  z ~ N(0, sigma_v^2)

    def step(self, grads):                                                  # identical to Adam from here on
        self.t += 1; b1, b2, t = self.b1, self.b2, self.t
        for p, g, m, v in zip(self.params, grads, self.m, self.v):
            m[:] = b1 * m + (1 - b1) * g
            v[:] = b2 * v + (1 - b2) * g * g
            p -= self.lr * (m / (1 - b1 ** t)) / (np.sqrt(v / (1 - b2 ** t)) + self.eps)

# Check of Lemma 1: with a zero gradient, the bias-corrected states after one step are 9*m0 and 999*v0.
w = np.zeros(5); opt = SADAM([w], seed=1); m0, v0 = opt.m[0].copy(), opt.v[0].copy()
opt.step([np.zeros(5)])
print("m_hat_1 / m0 =", np.round(opt.m[0] / (1 - 0.9) / m0, 6))
print("v_hat_1 / v0 =", np.round(opt.v[0] / (1 - 0.999) / v0, 6))

### Implicit warm-up
For gradients of typical size $s$, the initial $v_0$ dominates the denominator until roughly
$t^* = \ln(1+\sigma_v^2/s^2)/(1-\beta_2)$, so early steps are smaller than Adam's: an implicit, coordinate-wise warm-up.
Below, 2,000 coordinates receive noisy gradients of scale $s = 0.002$ (close to the measured gradient RMS of the paper's networks).

In [ ]:
import matplotlib.pyplot as plt
s, T, n = 0.002, 5000, 2000
rng = np.random.default_rng(0); G = s * (1 + 0.5 * rng.standard_normal((T, n)))
plt.figure(figsize=(7, 3.2))
for sv, lab in [(0.0, "Adam"), (0.003, r"SADAM $\sigma_v=0.003$"), (0.01, r"SADAM $\sigma_v=0.01$ (default)"), (0.03, r"SADAM $\sigma_v=0.03$")]:
    w = np.zeros(n); opt = SADAM([w], sigma_m=0.0, sigma_v=sv); steps = []
    for t in range(T):
        before = w.copy(); opt.step([G[t]]); steps.append(np.median(np.abs(w - before)) / opt.lr)
    line, = plt.plot(np.arange(1, T + 1), steps, label=lab)
    if sv > 0:
        plt.axvline(np.log(1 + sv**2 / s**2) / (1 - 0.999), color=line.get_color(), ls=":", lw=1)
plt.xscale("log"); plt.xlabel("iteration t"); plt.ylabel(r"median $|\Delta\theta_t|/\alpha$")
plt.title("Step size relative to Adam (dotted lines: predicted $t^*$)"); plt.legend(fontsize=8); plt.grid(alpha=.3); plt.show()

## 3. Regenerate the paper's tables and figures from the raw logs
The repository contains the per-run JSON logs of all 287 MNIST/CIFAR-10 runs and the 500-run UCI-digits study.

In [ ]:
import filecmp
for script in ["npanalyze.py", "replot.py", os.path.join("theory_and_digits", "digits_report.py")]:
    r = subprocess.run([sys.executable, script], cwd="code", capture_output=True, text=True)
    assert r.returncode == 0, r.stderr
same = [filecmp.cmp(f"outputs/tables/{f}", f"tables/{f}", shallow=False) for f in sorted(os.listdir("tables"))]
print(f"{sum(same)} of {len(same)} regenerated tables are identical to the tables in the paper.")

In [ ]:
import json
S = json.load(open("results/summary.json"))
NAMES = {"adam": "Adam", "nadam": "NAdam", "aadam": "AAdam", "adamw": "AdamW", "radam": "RAdam", "adabelief": "AdaBelief",
         "adam_warmup": "Adam + warm-up", "adam_v0rnd": "Adam-v0rnd", "sadamw": "SADAM-W", "sadam": "SADAM"}
EXP = [("E1", "MNIST / MLP"), ("E2", "CIFAR-10 / CNN"), ("E3", "CIFAR-10 / ResNet-8")]
print("Final test accuracy (%), mean ± s.d. over seeds   [epochs to reach the training-loss threshold]\n")
print(f"{'Optimizer':<16}" + "".join(f"{t:>28}" for _, t in EXP))
for k, name in NAMES.items():
    row = f"{name:<16}"
    for e, _ in EXP:
        d = S.get(f"{e}|{k}")
        row += f"{'—':>28}" if d is None else f"{d['acc']:>12.2f} ± {d['sd']:.2f}  [{d['ep']:>5.1f}]".rjust(28)
    print(row)

In [ ]:
from IPython.display import Image, Markdown, display
FIGS = [("fig_E1_curves", "MNIST / MLP: training loss and test accuracy"),
        ("fig_E2_curves", "CIFAR-10 / small CNN"),
        ("fig_E3_curves", "CIFAR-10 / ResNet-8"),
        ("fig_sens_mnist", "Sensitivity to sigma_m and sigma_v (MNIST)"),
        ("fig_steps_mnist", "Early-iteration step sizes (MNIST)"),
        ("fig_digits_sens", "UCI-digits sensitivity study (500 runs)")]
for f, cap in FIGS:
    display(Markdown(f"**{cap}**")); display(Image(f"outputs/figs/{f}.png", width=820))

## 4. Try it yourself: Adam vs SADAM on handwritten digits
MLP 64-128-64-10 on the UCI digits set (1,797 images). Same initial weights and mini-batch order for both optimizers (paired seeds).
Change the values and re-run the cell (about 30 s).

In [ ]:
SIGMA_M = 0.01   #@param {type:"number"}
SIGMA_V = 0.01   #@param {type:"number"}
EPOCHS = 30      #@param {type:"integer"}
SEEDS = 3        #@param {type:"integer"}

In [ ]:
import nn
from optim import Opt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

X, y = load_digits(return_X_y=True); X = X / 16.0
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-8
Xtr, Xte = ((Xtr - mu) / sd).astype(np.float32), ((Xte - mu) / sd).astype(np.float32)

def train(opt_name, seed, sigma_m=0.01, sigma_v=0.01, bs=32):
    rng = np.random.default_rng(seed)
    model = nn.Seq([nn.Linear(64, 128, rng), nn.ReLU(), nn.Linear(128, 64, rng), nn.ReLU(), nn.Linear(64, 10, rng)])
    opt = Opt(opt_name, model.params, lr=1e-3, seed=seed, sigma_m=sigma_m, sigma_v=sigma_v)
    order = np.random.default_rng(1000 + seed); loss_hist, acc_hist = [], []
    for ep in range(EPOCHS):
        perm = order.permutation(len(Xtr)); tot = 0.0
        for k in range(0, len(perm), bs):
            idx = perm[k:k + bs]
            for p in model.params: p.grad[...] = 0
            l, d = nn.softmax_ce(model.fwd(Xtr[idx]), ytr[idx]); model.bwd(d); opt.step(); tot += l * len(idx)
        loss_hist.append(tot / len(Xtr)); acc_hist.append((model.fwd(Xte, train=False).argmax(1) == yte).mean() * 100)
    return np.array(loss_hist), np.array(acc_hist)

runs = {"Adam": [train("adam", s) for s in range(SEEDS)],
        f"SADAM (σm={SIGMA_M:g}, σv={SIGMA_V:g})": [train("sadam", s, SIGMA_M, SIGMA_V) for s in range(SEEDS)]}
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4)); ep = np.arange(1, EPOCHS + 1)
for lab, rs in runs.items():
    L = np.array([r[0] for r in rs]); A = np.array([r[1] for r in rs])
    ax[0].plot(ep, L.mean(0), label=lab); ax[1].plot(ep, A.mean(0), label=lab)
    print(f"{lab:<32} final test accuracy {A[:, -1].mean():.2f} ± {A[:, -1].std():.2f} %   final training loss {L[:, -1].mean():.4f}")
ax[0].set_yscale("log"); ax[0].set_ylabel("training loss"); ax[1].set_ylabel("test accuracy (%)")
for a in ax: a.set_xlabel("epoch"); a.grid(alpha=.3); a.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 5. Optional: re-run real MNIST training and compare with the logged result
Downloads MNIST (about 17 MB) and trains the paper's MLP for a few epochs with Adam and SADAM (seed 0).
With single-threaded BLAS the numbers match the logs in `results/E1` exactly. About 1 minute.

In [ ]:
RUN_MNIST = True   #@param {type:"boolean"}
MNIST_EPOCHS = 3   #@param {type:"integer"}

In [ ]:
if RUN_MNIST and not os.environ.get("CI"):
    os.makedirs("data", exist_ok=True)
    if not os.path.exists("data/nndl/data/mnist.pkl.gz"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/mnielsen/neural-networks-and-deep-learning.git", "data/nndl"], check=True)
    env = dict(os.environ, OPENBLAS_NUM_THREADS="1", OMP_NUM_THREADS="1", MKL_NUM_THREADS="1")
    for o in ["adam", "sadam"]:
        subprocess.run([sys.executable, "run.py", "--dataset", "mnist", "--model", "mlp", "--opt", o, "--seed", "0",
                        "--epochs", str(MNIST_EPOCHS), "--bs", "256", "--out", "../outputs/mnist_rerun"], cwd="code", env=env, check=True)
        new = json.load(open(f"outputs/mnist_rerun/mnist_mlp_{o}_sm0.01_sv0.01_seed0.json"))["epochs"]
        old = json.load(open(f"results/E1/mnist_mlp_{o}_sm0.01_sv0.01_seed0.json"))["epochs"]
        for e in range(MNIST_EPOCHS):
            print(f"{o:<6} epoch {e+1}: train loss {new[e]['train_loss']:.6f} (logged {old[e]['train_loss']:.6f})   "
                  f"test acc {100*new[e]['test_acc']:.2f}% (logged {100*old[e]['test_acc']:.2f}%)")
else:
    print("Skipped.")